# Project 4 Live Demo：推論 + PD 控制（可切換模型）

流程：`camera.py` 取像 → `preprocessing.py`（把車道線標成紅色，和訓練時相同，固定開啟）→ `loader.py` 載入模型（`resnet18.py` / `mobilenet_v2.py`）並推論 → `control.py` PD + bias 差速控制。畫面顯示的就是模型實際看到的影像。

- 上方下拉選單切換 checkpoint（不同 augmentation / 有無 DepGraph 剪枝 / PyTorch 或 TensorRT）。
- 「允許移動」**預設關閉**：關閉時只推論、顯示，不送馬達。第一次請把車架高或抓好。
- 「輸入干擾」可以在即時畫面上加柏林噪聲 / 壓縮失真，測各模型的穩健性。
- 「停止」會立刻停車並取消移動許可；切換模型時也會自動停車並取消移動許可。

`models/` 放 `train.py` 產生的 `*.pth`（及同名 `.json`），以及板子上用 `trtexec` + `compression.wrap_trt_engine` 做出的 TRT `.pth`。

## 1. 設定

In [ ]:
import os, sys, time, threading, csv, datetime
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display

# Make the modules in project4/ importable
sys.path.insert(0, os.path.abspath('.'))
# Camera sources, tensor conversion, input disturbance and drawing helpers
import camera as cam_mod
# Checkpoint loading (PyTorch / TensorRT) and listing for the dropdown
from loader import load_model, list_models
# Lane marking: the same preprocessing the models were trained with
from preprocessing import apply_frame as mark_lanes
# PD + bias motor controller and output conversion
from control import PDController, model_output_to_xy

# Folder with the trained checkpoints
MODELS_DIR = 'models'
# Frame source: 'jetbot' | 'csi' (never queues old frames) | 'opencv' | 'folder' | 'dummy'
CAMERA_SOURCE = 'jetbot'
# Image folder replayed when CAMERA_SOURCE = 'folder'
CAMERA_FOLDER = 'data/dataset_xy_2026-10-21/dataset_xy'
# False: run without a JetBot (motor values are only displayed)
USE_ROBOT = True
# Folder for the CSV drive logs
LOG_DIR = 'runs'
# Drop frames queued during a model switch (jetbot source restarts its pipeline, ~1-2 s)
FLUSH_ON_SWITCH = True
# Max display updates per second; higher rates make the Wi-Fi / browser fall behind
DISPLAY_FPS = 15

# Run on the GPU when there is one
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Show the device and the checkpoints that were found
print('device:', device, '| models:', [os.path.basename(p) for _, p in list_models(MODELS_DIR)])

## 2. 相機與車子

In [ ]:
# Open the chosen frame source
camera = cam_mod.open_camera(CAMERA_SOURCE, folder=CAMERA_FOLDER)

# Stand-in for a jetbot motor when no robot is attached
class _FakeMotor(object):
    # Motor value written by the control loop
    value = 0.0

# Stand-in for jetbot.Robot with the same attributes the loop uses
class _FakeRobot(object):
    # Create two fake motors
    def __init__(self):
        # Left and right motor
        self.left_motor, self.right_motor = _FakeMotor(), _FakeMotor()
    # Set both motors to zero
    def stop(self):
        # Stop both motors
        self.left_motor.value = self.right_motor.value = 0.0

# Use the real robot when requested
if USE_ROBOT:
    from jetbot import Robot
    # Real motor driver
    robot = Robot()
# Otherwise simulate it
else:
    # Fake motors
    robot = _FakeRobot()
# Make sure the robot starts standing still
robot.stop()
# Controller; its gains are updated from the sliders every frame
controller = PDController()

## 3. 介面

In [ ]:
# Common layout for the parameter sliders
S = dict(layout=widgets.Layout(width='320px'), style={'description_width': '110px'})
# Common settings for the read-only value displays
R = dict(disabled=True, readout_format='.3f', **S)

# --- Variables ------------------------------------------------------------
# Model selector filled with the checkpoints in MODELS_DIR
model_dd = widgets.Dropdown(description='模型', options=list_models(MODELS_DIR),
                            layout=widgets.Layout(width='520px'), style={'description_width': '50px'})
# Button to rescan MODELS_DIR
refresh_btn = widgets.Button(description='重新掃描', layout=widgets.Layout(width='90px'))
# Information line about the loaded model (and errors)
model_info = widgets.HTML(value='(尚未載入)')
# Input disturbance: Perlin-noise lighting on the live frame
noise_perlin = widgets.Checkbox(description='輸入干擾：柏林噪聲', value=False, indent=False)
# Input disturbance: compression artefacts on the live frame
noise_comp = widgets.Checkbox(description='輸入干擾：壓縮失真', value=False, indent=False)

# --- Control parameters (ranges from slides P.71) -------------------------
# Base forward speed
speed_gain = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, description='speed gain', **S)
# Proportional steering gain
steer_gain = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='steering gain', **S)
# Derivative steering gain
steer_kd = widgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd',
                               readout_format='.3f', **S)
# Steering offset that compensates for unequal motors
steer_bias = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.0, description='steering bias', **S)
# Hairpin extension (not in the slides; 0 = slide formula): slow down in sharp turns
turn_slow = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description='轉彎減速 k', **S)
# Hairpin extension: allow the inner wheel to spin backwards
allow_rev = widgets.Checkbox(description='內側輪可反轉', value=False, indent=False)

# --- Live readings (slides P.73) ------------------------------------------
# Predicted x
x_r = widgets.FloatSlider(min=-1.0, max=1.0, description='x', **R)
# Predicted y (after the slide's conversion)
y_r = widgets.FloatSlider(min=-1.0, max=1.0, description='y', **R)
# Steering angle in radians
angle_r = widgets.FloatSlider(min=-1.6, max=1.6, description='angle (rad)', **R)
# Total steering value
steer_r = widgets.FloatSlider(min=-1.0, max=1.0, description='steering', **R)
# Speed actually used (after turn slowdown)
speed_r = widgets.FloatSlider(min=0.0, max=1.0, description='實際 speed', **R)
# Left motor value
left_r = widgets.FloatSlider(min=-1.0, max=1.0, description='left motor', **R)
# Right motor value
right_r = widgets.FloatSlider(min=-1.0, max=1.0, description='right motor', **R)
# FPS, latency, P / D terms and moving state
stats = widgets.HTML()

# --- Operation ------------------------------------------------------------
# Safety switch: motors only receive values while this is ticked
allow_move = widgets.Checkbox(description='允許移動（送馬達）', value=False, indent=False)
# Record every control step to a CSV file
record = widgets.Checkbox(description='錄製 CSV', value=False, indent=False)
# Start the inference loop
start_btn = widgets.Button(description='開始推論', button_style='success')
# Stop the loop and the robot
stop_btn = widgets.Button(description='停止', button_style='danger')
# Live image (what the model sees, with the prediction drawn on it)
image_w = widgets.Image(format='jpeg', width=336, height=336)

# Arrange all widgets: model row, image + controls, operation buttons
ui = widgets.VBox([
    widgets.HBox([model_dd, refresh_btn]), model_info,
    widgets.HBox([
        widgets.VBox([image_w, widgets.HBox([noise_perlin, noise_comp])]),
        widgets.VBox([widgets.HTML('<b>控制參數</b>'), speed_gain, steer_gain, steer_kd, steer_bias,
                      widgets.HTML('<b>髮夾彎（預設關閉）</b>'), turn_slow, allow_rev,
                      widgets.HTML('<b>即時讀數</b>'), x_r, y_r, angle_r, steer_r, speed_r, left_r, right_r, stats]),
    ]),
    widgets.HBox([start_btn, stop_btn, allow_move, record]),
])
# Show the interface
display(ui)

## 4. 推論 / 控制迴圈

In [ ]:
# Shared state between the UI callbacks and the loop thread
state = {'running': False, 'thread': None, 'model': None, 'pending': None, 'error': None,
         'csv': None, 'writer': None, 'closed': False}

# HTML summary of a loaded model for the info line
def describe(m):
    # Metadata from the checkpoint's .json
    meta = m.meta or {}
    # Start with file name, runtime and dtype
    parts = ['<b>%s</b>' % m.describe()]
    # Add the label (network | augmentation | compression)
    if meta.get('label'): parts.append(meta['label'])
    # Preprocessing the model was trained with
    pre = meta.get('preprocessing') or {}
    # Warn if the model was not trained on lane-marked images
    if pre.get('method') != 'lane_marking':
        # Red warning text
        parts.append('<span style="color:red">這個模型不是用車道線標示的資料訓練的</span>')
    # Parameter count
    if meta.get('params'): parts.append('params %.2fM' % (meta['params'] / 1e6))
    # Multiply-accumulate count
    if meta.get('macs'): parts.append('MACs %.2fG' % (meta['macs'] / 1e9))
    # Best validation loss
    if meta.get('best_val_loss') is not None: parts.append('val loss %.4f' % meta['best_val_loss'])
    # Loss on the held-out test folder (only for models trained on split folders)
    if meta.get('test_loss') is not None: parts.append('test loss %.4f' % meta['test_loss'])
    # One line separated by bars
    return ' | '.join(parts)

# Stop the motors and show zero motor values
def halt():
    # Stop both motors
    robot.stop()
    # Reset the motor readings
    left_r.value = right_r.value = 0.0

# Open or close the CSV drive log
def set_recording(on):
    # Recording switched on and no file open yet
    if on and state['csv'] is None:
        # Create the log folder if needed
        os.makedirs(LOG_DIR, exist_ok=True)
        # File name with the current date and time
        path = os.path.join(LOG_DIR, 'drive_%s.csv' % datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
        # Open the file
        state['csv'] = open(path, 'w', newline='')
        # CSV writer for it
        state['writer'] = csv.writer(state['csv'])
        # Header row
        state['writer'].writerow(['t', 'model', 'moving', 'perlin', 'compression', 'x', 'y', 'angle',
                                  'p', 'd', 'steering', 'speed', 'speed_eff', 'left', 'right', 'latency_ms',
                                  'kp', 'kd', 'bias', 'turn_slowdown', 'reverse'])
        # Tell the user where the log goes
        print('recording ->', path)
    # Recording switched off while a file is open
    elif not on and state['csv'] is not None:
        # Close the file
        state['csv'].close()
        # Forget the file and the writer
        state['csv'] = state['writer'] = None

# Inference + control loop, runs in a background thread
def loop():
    # FPS measurement: window start, frames in window, last FPS
    fps_t, fps_n, fps = time.time(), 0, 0.0
    # Smoothed latency (None until the first frame)
    lat_avg = None
    # Time of the last display update
    disp_t = 0.0
    # Any error stops the loop and is shown in the UI
    try:
        # Run until stop is pressed
        while state['running']:
            # Switch models inside the loop thread (same CUDA / TensorRT context).
            # Stop the robot and untick "allow move" while switching; tick it again by hand
            # once the new model's predictions look right.
            if state['pending'] is not None:
                # Take the requested checkpoint and clear the request
                path, state['pending'] = state['pending'], None
                # Untick "allow move" for safety
                allow_move.value = False
                # Stop the motors
                halt()
                # Show that loading is in progress
                model_info.value = '載入中… %s' % os.path.basename(path)
                # No model while loading
                state['model'] = None
                # Load the checkpoint
                m = load_model(path, device)
                # Warm-up: the first TRT / CUDA inferences are slow, so do them before the flush
                warm = camera.read()
                # Only possible once the camera has a frame
                if warm is not None:
                    # A few warm-up runs
                    for _ in range(3):
                        # Same input path as the real loop
                        m(cam_mod.preprocess(mark_lanes(warm), device, m.dtype))
                # Throw away frames queued during loading so the picture is live again
                if FLUSH_ON_SWITCH:
                    # Show that the queue is being emptied
                    model_info.value = '清空相機佇列… %s' % os.path.basename(path)
                    # Empty the camera queue
                    camera.flush()
                # Start using the new model
                state['model'] = m
                # Forget the previous angle (D term)
                controller.reset()
                # Restart latency averaging
                lat_avg = None
                # Restart FPS measurement
                fps_t, fps_n, fps = time.time(), 0, 0.0
                # Show the model summary
                model_info.value = describe(m)
            # Current model
            m = state['model']
            # Latest camera frame
            frame = camera.read()
            # Nothing to do without a frame or a model
            if frame is None or m is None:
                # Wait a little
                time.sleep(0.02)
                # Try again
                continue
            # Optional input disturbance (before lane marking, as in training)
            frame = cam_mod.degrade(frame, noise_perlin.value, noise_comp.value)

            # Start of the timed part (preprocessing + inference)
            t0 = time.time()
            # Preprocessing: paint the lane lines red (same as training)
            frame = mark_lanes(frame)
            # Run the network
            out = m(cam_mod.preprocess(frame, device, m.dtype))
            # Prediction as a flat float numpy array
            xy = out.detach().float().cpu().numpy().flatten()
            # Wait for the GPU so the timing is correct
            if device.type == 'cuda':
                # Block until all GPU work is done
                torch.cuda.synchronize()
            # Latency in milliseconds
            lat = (time.time() - t0) * 1000.0
            # Exponential moving average of the latency
            lat_avg = lat if lat_avg is None else 0.9 * lat_avg + 0.1 * lat

            # Copy speed and P gain from the sliders
            controller.speed, controller.kp = speed_gain.value, steer_gain.value
            # Copy D gain and bias from the sliders
            controller.kd, controller.bias = steer_kd.value, steer_bias.value
            # Copy the hairpin options
            controller.turn_slowdown, controller.allow_reverse = turn_slow.value, allow_rev.value
            # Network output -> (x, y) for the angle
            x, y = model_output_to_xy(xy)
            # One control step
            left, right = controller.update(x, y)
            # All intermediate values of this step
            c = controller.last
            # Drive only when "allow move" is ticked
            if allow_move.value:
                # Send the motor values
                robot.left_motor.value, robot.right_motor.value = left, right
            # Otherwise keep the robot still
            else:
                # Stop the motors
                robot.stop()

            # Count this frame
            fps_n += 1
            # Update the FPS once per second
            if time.time() - fps_t >= 1.0:
                # Frames / elapsed time, then start a new window
                fps, fps_t, fps_n = fps_n / (time.time() - fps_t), time.time(), 0

            # Control runs every frame; the display at most DISPLAY_FPS times per second
            # so updates do not pile up in the browser
            now = time.time()
            # Time for a display update
            if now - disp_t >= 1.0 / DISPLAY_FPS:
                # Remember when it happened
                disp_t = now
                # Show x and y
                x_r.value, y_r.value = x, y
                # Show angle and (clamped) steering
                angle_r.value, steer_r.value = c['angle'], max(-1.0, min(1.0, c['steering']))
                # Show effective speed and motor values
                speed_r.value, left_r.value, right_r.value = c['speed_eff'], left, right
                # Show the lane-marked frame with prediction and steering bar
                image_w.value = cam_mod.to_jpeg(cam_mod.draw_overlay(frame, xy, c['steering']))
                # Show FPS, latency, P / D terms and moving state
                stats.value = ('FPS <b>%.1f</b> | 前處理+推論 <b>%.1f ms</b>（平均 %.1f）| P %.3f D %.3f | %s'
                               % (fps, lat, lat_avg, c['p'], c['d'],
                                  '<span style="color:red">移動中</span>' if allow_move.value else '未送馬達'))
            # Open / close the log to follow the checkbox
            set_recording(record.value)
            # Write one log row while recording
            if state['writer'] is not None:
                # Time, model, switches, readings, latency and gains
                state['writer'].writerow(['%.3f' % time.time(), os.path.basename(m.path), int(allow_move.value),
                                          int(noise_perlin.value), int(noise_comp.value)]
                                         + ['%.4f' % c[k] for k in ('x', 'y', 'angle', 'p', 'd', 'steering',
                                                                    'speed', 'speed_eff', 'left', 'right')]
                                         + ['%.2f' % lat, controller.kp, controller.kd, controller.bias,
                                            controller.turn_slowdown, int(controller.allow_reverse)])
    # Any error inside the loop
    except Exception as e:
        import traceback, html
        # Keep the error for inspection
        state['error'] = e
        # Unload the model
        state['model'] = None
        # Show the error and the traceback in the info line
        model_info.value = ('<span style="color:red"><b>推論迴圈因錯誤停止：%s</b></span>'
                            '<pre style="font-size:11px;line-height:1.2">%s</pre>'
                            % (html.escape(repr(e)), html.escape(traceback.format_exc())))
    # Always leave the robot in a safe state
    finally:
        # Stop the motors
        halt()
        # Close the log
        set_recording(False)
        # Mark the loop as stopped
        state['running'] = False

# "Start" button handler
def on_start(_=None):
    # Already running
    if state['running']:
        # Nothing to do
        return
    # The camera was released by the end button
    if state['closed']:
        # Ask for a kernel restart
        model_info.value = '<span style="color:red">相機已釋放，請 Restart Kernel 後重新執行。</span>'
        # Cannot start
        return
    # Load the selected model if none is loaded yet
    if state['model'] is None and model_dd.value:
        # Request it; the loop thread loads it
        state['pending'] = model_dd.value
    # Mark as running and clear the last error
    state['running'], state['error'] = True, None
    # Background thread running the loop
    state['thread'] = threading.Thread(target=loop, daemon=True)
    # Start it
    state['thread'].start()

# "Stop" button handler
def on_stop(_=None):
    # Untick "allow move"
    allow_move.value = False
    # Ask the loop to end
    state['running'] = False
    # Stop the motors right away
    halt()

# Dropdown change handler
def on_model_change(change):
    # A model was selected
    if change['new']:
        # Request it; the loop thread loads it
        state['pending'] = change['new']

# "Rescan" button handler
def on_refresh(_=None):
    # Re-read the checkpoints in MODELS_DIR
    model_dd.options = list_models(MODELS_DIR)

# Connect the start button
start_btn.on_click(on_start)
# Connect the stop button
stop_btn.on_click(on_stop)
# Connect the rescan button
refresh_btn.on_click(on_refresh)
# React to dropdown changes
model_dd.observe(on_model_change, names='value')

## 5. 推論速度比較（報告用）

在**車子不動**的情況下，對 `models/` 內每個模型用同一張畫面量 latency，條件相同才能比較（P.80）。先按「停止」再執行。

In [ ]:
# Measure the inference latency of every checkpoint on the same frame (robot standing still)
def benchmark(n=100, warmup=20):
    # The loop must not be using the GPU / camera at the same time
    assert not state['running'], '先按「停止」'
    # One frame shared by all models
    frame = camera.read()
    # (latency, label, runtime) per model
    rows = []
    # Every checkpoint in MODELS_DIR
    for label, path in list_models(MODELS_DIR):
        # Load it
        m = load_model(path, device)
        # Same input as in the loop (lane-marked)
        x = cam_mod.preprocess(mark_lanes(frame), device, m.dtype)
        # Warm-up runs, not timed
        for _ in range(warmup):
            # Run the network
            m(x)
        # Make sure the warm-up finished on the GPU
        if device.type == 'cuda': torch.cuda.synchronize()
        # Start the timer
        t0 = time.time()
        # Timed runs
        for _ in range(n):
            # Run the network
            m(x)
        # Wait for the GPU before stopping the timer
        if device.type == 'cuda': torch.cuda.synchronize()
        # Mean milliseconds per inference
        ms = (time.time() - t0) * 1000.0 / n
        # Store the result
        rows.append((ms, label, m.kind))
        # Free the model before loading the next one
        del m
    # Print fastest first
    for ms, label, kind in sorted(rows):
        # Latency, FPS, runtime and label
        print('%8.2f ms  %6.1f FPS  [%s]  %s' % (ms, 1000.0 / ms, kind, label))
    # Results for further use
    return rows

# Uncomment the next line to run the benchmark
# benchmark()

## 6. 結束：停車並釋放相機

做成按鈕，**只有手動按下才會執行**。如果直接寫成 cell，Run All 會一路跑到這裡，相機剛開就被關掉。

In [ ]:
# Button that stops everything and releases the camera
end_btn = widgets.Button(description='結束：停車並釋放相機', button_style='warning',
                         layout=widgets.Layout(width='220px'))

# "End" button handler
def on_end(_=None):
    # Stop the loop and the robot
    on_stop()
    # Prevent restarting with a released camera
    state['closed'] = True
    # Give the loop thread time to finish
    time.sleep(0.2)
    # Release the camera for other notebooks
    camera.stop()
    # The button can only be used once
    end_btn.disabled = True
    # Show that the camera is free
    end_btn.description = '相機已釋放'

# Connect the end button
end_btn.on_click(on_end)
# Show it
display(end_btn)